# Xenium–MACSima Single-Core Co-Registration

This notebook demonstrates how to co-register a single tissue core imaged by two
complementary spatial technologies:

| Technology | What it measures | Pixel size |
|---|---|---|
| **10x Xenium** | Targeted spatial transcriptomics (gene expression per cell) | 212.5 nm |
| **MACSima** | Highly multiplexed cyclic protein imaging (50+ markers) | 170 nm |

By aligning these modalities on the same tissue core, every cell gains both a
**transcriptomic profile** and a **protein expression profile** — enabling
multi-modal single-cell analysis at subcellular resolution.

## Workflow

1. Both instruments image the same TMA (tissue microarray) slide independently, producing separate coordinate systems.
2. We use SIFT-based feature matching to find the affine transform that maps the MACSima DAPI image onto the
   Xenium morphology-focus nuclear channel (the *target*).
3. The recovered affine is applied to all MACSima protein channels, bringing them into Xenium's coordinate space.
4. We use the Xenium cekll segmentation to measure the mean protein intensity from the registered MACSima channels.
5. The result is a `SpatialData` object containing aligned images, cell + nucleus
   segmentation boundaries, the transcriptomic and proteomic cell table and raw transcript coordinates

In [ ]:
import os
import warnings
import spatialdata_plot
import spatialdata as sd
import anndata as ad
# Registration workflow
from xenium_macsima_core_registration.xenium_macsima_core_registration import XeniumMacsimaRegistration, preprocess_reg

## Paths and Parameters

Two key identifiers connect the same tissue core across the two instruments:

- **POSITION** — the Xenium TMA core index (a float-valued string such as `'1.0'`)
- **ROI_INDEX** — the matching MACSima region-of-interest number

The mapping between positions and ROIs was established manually by comparing
the physical slide layout with each instrument's scan pattern

In [ ]:
# ── Target core ───────────────────────────────────────────────────────────────
POSITION = 1 
ROI_INDEX = 40      # Matching MACSima ROI number

# ── Registration scale factor ─────────────────────────────────────────────────
# Both SIFT images are down-scaled by this factor before feature detection.
SCALE_FACTOR = 3

# ── Input path (all under the local data/ directory) ─────────────────────────
DATA_DIR   = './data'

# MACSima image folder for this ROI
MACSIMA_FOLDER = f'{DATA_DIR}/macsima/ROI{ROI_INDEX}/'

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_ZARR = f'./aligned_objects/Slide_TMAN2-1-1/Pos_{POSITION}_ROI_{ROI_INDEX}.zarr'

print(f"Target core  : Xenium position {POSITION}  →  MACSima ROI {ROI_INDEX}")
print(f"Scale factor : {SCALE_FACTOR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Output       : {OUTPUT_ZARR}")

## Step 2 — Load the pre-cropped Xenium data (postition 1)

After this step the `registration` object is in exactly the same state as it
would be after `load_xenium_data()` + `align_data_tables()` + `calculate_roi_bbox()` + `crop()`.  
So we are starting the workflow from an already cropped Xenium 

In [ ]:
warnings.filterwarnings('ignore')
# ── Reads the Pre-Cropped Xenium and Writes it out as the base for the new aligned SpatialData object ───────────────────────────────────────────────────
registration = XeniumMacsimaRegistration(scale_factor=SCALE_FACTOR)
registration.aligned_data = sd.read_zarr(os.path.join(DATA_DIR, "xenium_01.zarr"))
registration.write_crop(OUTPUT_ZARR)

## Step 3 — Load the MACSima Image (ROI 40)

Two images serve as registration inputs:

| Image | Role |
|---|---|
| **Xenium morphology focus (ROI crop)** | *Target* — align **to** this image |
| **MACSima DAPI** | *Source* — align **from** this image | 

In [ ]:
warnings.filterwarnings('ignore')
# ── MACSima IMAGEs ──────────────────────────────────────────────────────────────
# Exclude background frames (marked with '_S_' for single vs '_B_' for background).
macsima_img = registration.load_macsima_images(MACSIMA_FOLDER)
print(f"MACSima image shape: {macsima_img.shape}")

## Step 4 — Image Registration

We use SIFT (Scale-Invariant Feature Transform) feature matching to estimate the
affine transform that maps the MACSima DAPI image onto the Xenium morphology-focus
nuclear channel (the *target*).

**Pre-processing pipeline (same for both images):**
1. Down-scale by `SCALE_FACTOR` — speeds up detection and reduces photon noise
2. CLAHE contrast enhancement (`skimage.exposure.equalize_adapthist`)
3. Normalise to uint8 range

The affine matrix is applied to the entire MACSima image stack with
`dask_image.ndinterp.affine_transform`.

In [ ]:
print("Running SIFT registration (from MACSima DAPI to Xenium DAPI)")
affine, aligned_macsima = registration.register_images()
registration.add_macsima()
print("Registration complete, writing the registered MACSima image.")
registration.write_macsima()
print("Writing the registered MACSima image complete.")

## Step 5 — Protein Quantification

| Layer                   | Type   |                                                       Description |
|---                      |---     |                                                                ---|
| `morphology_focus`      | Image  | Xenium nuclear channel (down-scaled), for structural context      |
| `macsima`               | Image  | Aligned MACSima image                                             |
| `cell_boundaries`       | Shapes | Xenium cell segmentation polygons                                 |
| `nucleus_boundaries`    | Shapes | Xenium nucleus segmentation polygons                              |
| `cell_labels`           | Labels | Xenium cell segmentation labels                                   |
| `rna_protein_table`     | Table  | AnnData cell table with protein and RNA expression                |
| `table`                 | Table  | AnnData cell table with RNA expression                            |
| `macsima_table`         | Table  | AnnData cell table with protein expression                        |
| `transcripts`           | Points | Individual transcript coordinates                                 |

All spatial layers share the `global` coordinate system.

In [ ]:
print("Quantifying proteins")
registration.quantify_proteins()
print("Writing the new RNA/protein table to the SpatialData object on disk")
registration.write_proteins()

In [ ]:
print(registration.aligned_data)

The .zarr file can be viewed with napari (using the napari-spatialdata plugin)
https://spatialdata.scverse.org/projects/napari/en/latest/index.html

## Step 6 — Registration Quality Check

We make a plot to visually verify the alignment.

**Panel A** shows the Xenium DAPI channel.

**Panel B** shows the original unaligned MACSima DAPI.

**Panel C** shows the aligned MACSima DAPI with the Xenium cell boundaries in red.

**Panel D** overlay of Xenium morphology focus DAPI (red) and aligned MACSima DAPI (green)

In [ ]:
# Quality control plot
figure = registration.create_registration_summary_plot()
figure.savefig(f'POS_{POSITION}_registration_summary.pdf', dpi=600)